In [1]:
import pandas as pd
from IPython.core.magics import display

json_path = '../process/raw_literature.json'


# Read JSON directly into a pandas DataFrame
df_json = pd.read_json(json_path, encoding='utf-8')
print("DataFrame loaded successfully:")

DataFrame loaded successfully:


In [2]:
df_json.info()

<class 'pandas.DataFrame'>
RangeIndex: 27288 entries, 0 to 27287
Data columns (total 49 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              27288 non-null  str    
 1   doi                             27288 non-null  str    
 2   title                           27288 non-null  str    
 3   display_name                    27288 non-null  str    
 4   publication_year                27288 non-null  int64  
 5   publication_date                27288 non-null  str    
 6   ids                             27288 non-null  object 
 7   language                        26777 non-null  str    
 8   primary_location                27288 non-null  object 
 9   type                            27288 non-null  str    
 10  indexed_in                      27288 non-null  object 
 11  open_access                     27288 non-null  object 
 12  authorships                     27288 non-n

In [3]:
df_json.head()

,id,doi,title,display_name,publication_year,publication_date,ids,language,primary_location,type,...,funders,has_content,content_urls,referenced_works_count,referenced_works,related_works,abstract_inverted_index,counts_by_year,updated_date,created_date
0,https://openalex.org/W2134954483,https://doi.org/,Global data on visual impairment in the year 2...,Global data on visual impairment in the year 2...,2004,2004-11-01,{'openalex': 'https://openalex.org/W2134954483...,en,"{'id': 'pmid:15640920', 'is_oa': False, 'landi...",article,...,[],"{'pdf': False, 'grobid_xml': False}",None,16,"[https://openalex.org/W105318348, https://open...","[https://openalex.org/W2412714437, https://ope...","{'This': [0], 'paper': [1], 'presents': [2], '...","[{'year': 2026, 'cited_by_count': 1}, {'year':...",2026-08-03T07:22:36.454288,2025-10-10T00:00:00
1,https://openalex.org/W2158135241,https://doi.org/,The global burden of oral diseases and risks t...,The global burden of oral diseases and risks t...,2005,2005-09-01,{'openalex': 'https://openalex.org/W2158135241...,en,"{'id': 'pmid:16211157', 'is_oa': False, 'landi...",article,...,[],"{'pdf': False, 'grobid_xml': False}",None,29,"[https://openalex.org/W177451543, https://open...","[https://openalex.org/W2030231118, https://ope...","{'This': [0], 'paper': [1], 'outlines': [2], '...","[{'year': 2026, 'cited_by_count': 2}, {'year':...",2026-08-01T09:00:35.917206,2025-10-10T00:00:00
2,https://openalex.org/W2130275438,https://doi.org/,Counting the dead and what they died from: an ...,Counting the dead and what they died from: an ...,2005,2005-03-01,{'openalex': 'https://openalex.org/W2130275438...,en,"{'id': 'pmid:15798840', 'is_oa': False, 'landi...",article,...,[],"{'pdf': False, 'grobid_xml': False}",None,9,"[https://openalex.org/W1527672443, https://ope...","[https://openalex.org/W2103654194, https://ope...","{'OBJECTIVE:': [0], 'We': [1, 25], 'sought': [...","[{'year': 2025, 'cited_by_count': 18}, {'year'...",2026-08-03T07:22:36.454288,2025-10-10T00:00:00
3,https://openalex.org/W2115767265,https://doi.org/,Re-evaluating the burden of rabies in Africa a...,Re-evaluating the burden of rabies in Africa a...,2005,2005-05-01,{'openalex': 'https://openalex.org/W2115767265...,en,"{'id': 'pmid:15976877', 'is_oa': False, 'landi...",article,...,[],"{'pdf': True, 'grobid_xml': True}",{'pdf': 'https://content.openalex.org/works/W2...,38,"[https://openalex.org/W20447875, https://opena...","[https://openalex.org/W2381968804, https://ope...","{'OBJECTIVE:': [0], 'To': [1], 'quantify': [2]...","[{'year': 2025, 'cited_by_count': 33}, {'year'...",2026-07-15T18:14:33.161393,2025-10-10T00:00:00
4,https://openalex.org/W2143970799,https://doi.org/,The treatment gap in mental health care.,The treatment gap in mental health care.,2004,2004-11-01,{'openalex': 'https://openalex.org/W2143970799...,en,"{'id': 'pmid:15640922', 'is_oa': False, 'landi...",article,...,[],"{'pdf': False, 'grobid_xml': False}",None,98,"[https://openalex.org/W39011151, https://opena...","[https://openalex.org/W2114847905, https://ope...","{'Mental': [0], 'disorders': [1, 22, 92, 133, ...","[{'year': 2026, 'cited_by_count': 2}, {'year':...",2026-06-11T09:08:48.828518,2025-10-10T00:00:00


In [4]:
# Filter out records where abstract_inverted_index is missing or empty
df_filtered = df_json[
    df_json['abstract_inverted_index'].notna() &
    (df_json['abstract_inverted_index'].str.len() > 0)
].copy()

def reconstruct_abstract(inverted_index):
    # Reconstruct text from OpenAlex inverted index dictionary
    if not isinstance(inverted_index, dict) or not inverted_index:
        return ""

    word_positions = []
    for word, positions in inverted_index.items():
        for pos in positions:
            word_positions.append((pos, word))

    word_positions.sort(key=lambda x: x[0])
    return " ".join([word for pos, word in word_positions])

# Convert inverted index to plain text abstract
df_filtered['abstract'] = df_filtered['abstract_inverted_index'].apply(reconstruct_abstract)

# Save the processed dataset to JSON and CSV formats for subsequent LLM operations
df_filtered.to_json('processed_literature.json', orient='records', force_ascii=False)

print(f"Original dataset count: {len(df_json)}")
print(f"Filtered dataset count: {len(df_filtered)}")

Original dataset count: 27288
Filtered dataset count: 17436
